In [ ]:
%run fabric-domain-management-util

In [ ]:
display(domain_intended)

In [ ]:
display(workspace_domain_intended)

## Domain configuration section

In [ ]:
# check for validity
desired_domain_validity = check_domain_validity(domain_intended)
# make a useful error value
desired_domain_invalid_dict = {
  "message": "Input Domain file is invalid",
  "invalid_domains": desired_domain_validity
}
# crash notebook if inputs aren't valid
if desired_domain_validity != None:
    mssparkutils.notebook.exit(desired_domain_invalid_dict)


In [ ]:
# generate delta between desired and actual
# X = desired, Y = current
delta_df = pd.merge(domain_intended, df_domains_current, on=['Domain ID'], how='outer',indicator=True)
# split the adds and removes into their own little DFs
to_add = delta_df[delta_df['_merge'] == "left_only"]
to_add = to_add.iloc[:, 0:5]
to_add.columns = domain_config_column_names
# to move delete
to_remove = delta_df[delta_df['_merge'] == "right_only"]
# ones needing updates
already_present = delta_df[delta_df['_merge'] == "both"]
## check for differences
updates_not_needed = np.array(already_present['Description_x'] == already_present['Description_y']) & np.array(already_present['Description_x'] == already_present['Description_y'])
already_present.insert(loc=0, column = "matched", value = updates_not_needed)
to_adjust = already_present[already_present['matched'] == False]

In [ ]:
update_domains(to_adjust)
delete_domains(to_remove)
add_domains_refuse(to_add)

## Workspace assignment section

In [ ]:
# loop through current state via API, and generate a finalised dataframe of current state
domain_workspace_current = pd.DataFrame(columns=domain_workspace_column_names)
with labs.service_principal_authentication(
        key_vault_uri = key_vault_uri,
        key_vault_tenant_id = 'tenant-fabric-sp',
        key_vault_client_id = 'fabric-admin-sp-app-id', 
        key_vault_client_secret = admin_sp_client_secret_name[1]
        ):
    for index, row in df_domains_current.iterrows():
        tempdf = labs.admin.list_domain_workspaces(row['Domain ID'])
        tempdf.insert(0, "Domain ID", row['Domain ID'])
        domain_workspace_current = pd.concat([domain_workspace_current, tempdf])
domain_workspace_current.drop(columns=['Workspace Name'], inplace=True)
domain_workspace_current.sort_values(by = domain_workspace_column_names, inplace=True)

In [ ]:
delta_df = pd.merge(workspace_domain_intended, domain_workspace_current, how='outer', on = 'Workspace ID',indicator=True)

# generate and clean what needs to be added due to currently lacking an assignment
to_add = delta_df[delta_df['_merge'] == "left_only"]
to_add = to_add.iloc[:,0:2]
to_add.columns = domain_workspace_column_names

# generate what needs to be re-assigned
both = delta_df[delta_df['_merge'] == "both"]
to_move = both[both['Domain ID_x'] != both['Domain ID_y']]

# generate and clean what needs to be removed - completely de-assigned
to_remove = delta_df[delta_df['_merge'] == "right_only"]

In [ ]:
## start processing changes
# do we need to relocate any workspaces? 
domain_workspace_move(to_move)
domain_workspace_assign(to_add)
domain_workspace_detach(to_remove)

print('end all processes')

In [ ]:
notebookutils.notebook.exit("Success")